# Web scraping: collect text and check the results

Web scraping means downloading a web page and extracting information from it. Today we collect paragraph text from a small list of energy-related websites.

Our questions:

1. How can we extract text from one page?
2. How can we repeat this for a list of URLs?
3. How do we record what failed and check whether the extracted text is useful?

We first work on one page. Then we repeat the same steps in a loop. **Every URL must appear in our results, including the ones that fail.**

Run cells with **Shift + Enter**. Complete the cells with `# ...` comments; open **Show solution** if you need help.

## 1. Open our tools

`requests` downloads a page. `BeautifulSoup` lets us search its HTML. `pandas` stores the results as a table. `time` lets us pause between requests.

If an import fails, open a terminal in your course project folder and run `uv add requests beautifulsoup4 pandas openpyxl`. Then restart the notebook kernel and run from the top. The package is called `beautifulsoup4`, but we import it from `bs4`.

For larger projects, check the site's terms and `robots.txt`, keep the number of requests low, and look for a downloadable dataset or API. If access is blocked, record it and choose another source.

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

## 2. Make our URL list

The spreadsheet `url_collection.xlsx` contains the selected page titles and URLs. Keep it beside this notebook. We read it with the same `read_excel()` used in the pandas lectures.

In [ ]:
sources = pd.read_excel("url_collection.xlsx")
sources

Most of these are homepages; they will give us introductory text rather than one complete article. The spreadsheet also includes an energy overview from Our World in Data and a solar-energy page from the European Commission. All ten URLs are stored in the same file.

The Swiss Energy Program website is a private project, distinct from the federal SwissEnergy programme on the BFE website. Its root address returns Italian to a plain Python request. `/en/` redirects to that root address; the language selector links to `/en/homepage` instead.

A browser can send language preferences or remember a language choice in cookies. Python does not automatically inherit them. Even on the English page, the site's cookie notice is in Italian, so read the extracted text before using it.

In [ ]:
urls = sources["Url"].tolist()
urls

`tolist()` turns the URL column into a Python list. We now have ten URLs. We will test the same list with the same extraction recipe, even though the websites have different structures.

## 3. Download one page

We start with the Swiss Federal Office of Energy, the third URL (position 2).

`timeout=20` limits how long `requests` waits for a connection or for data to arrive. It prevents an unresponsive site from holding up the notebook indefinitely.

In [ ]:
url = urls[2]
print(url)
response = requests.get(url, timeout=20)
print(response.status_code)
print(response.url)

### Check the response

| Status | Usual meaning |
|---|---|
| 200 | The server returned a page. We still need to check its content. |
| 403 | The server refused access to this request. |
| 404 | The requested page was not found. |
| 429 | Too many requests. Stop and follow the site's instructions. |
| 500 or above | A server error occurred. |

`response.url` shows the final address after any redirects. `raise_for_status()` raises an error for HTTP error responses. With one page, stopping helps us notice a problem; later we will catch that error and continue to the next URL.

If this first page fails on your connection, try the Our World in Data or European Commission URL from the end of the list and rerun from this section.

In [ ]:
response.raise_for_status()
print(response.headers.get("Content-Type", ""))
print(response.text[:500])

## 4. Find paragraph text in the HTML

A browser renders HTML into a page. `requests` downloads the response, and does not run JavaScript. A page can therefore look complete in your browser while the downloaded HTML contains less information.

HTML uses tags. `<p>` marks a paragraph; `<main>` often marks the main content. Beautiful Soup searches these tags.

`response.content` contains the downloaded bytes. Beautiful Soup reads them using Python's built-in `"html.parser"`.

### Explore the page in your browser

Before we select tags in Python, look at the page's HTML in the browser.

1. Open the [SwissEnergy programme page](https://www.bfe.admin.ch/en/the-swissenergy-programme) in Chrome, Edge or Firefox.
2. Right-click a paragraph in the programme description and choose **Inspect**. On a Mac, Control-click also opens the context menu.
3. A developer-tools panel opens beside the page. In Chrome and Edge, the HTML tree is in **Elements**; in Firefox, it is in **Inspector**. The selected element might be a `<p>` or something inside it, such as a link.
4. Look at the enclosing elements. Expand and collapse them to see which elements contain others. Move your pointer over an element in the HTML tree: the browser highlights the corresponding part of the page.
5. Click inside the HTML tree and search for `main` using **Ctrl + F** on Windows or **Command + F** on Mac. Select the `<main>` element and see which part of the page it contains.

**Discuss:** does `<main>` contain the programme description? Does it also contain related links? Where are the navigation and footer?

### What do the tags mean?

| Tag | What it usually contains |
|---|---|
| `<body>` | The page content, including navigation and footer. |
| `<main>` | The main content of the page. |
| `<article>` | A self-contained article or other piece of content. |
| `<h1>` | A top-level heading. |
| `<p>` | A paragraph. |
| `<div>` | A general container that groups other elements. |
| `<nav>` | Navigation links. |
| `<footer>` | Footer information. |

Here is a simplified example, not an exact copy of the website:

```html
<body>
    <nav>Home | About | Contact</nav>
    <main>
        <h1>The SwissEnergy programme</h1>
        <div>
            <p>This is a paragraph about the programme.</p>
            <p>This is another paragraph.</p>
        </div>
    </main>
    <footer>Contact details</footer>
</body>
```

The paragraphs are **inside** `<main>`, even though a `<div>` sits between them. In Python, `soup.find("main")` selects that container. Calling `.find_all("p")` on it finds the paragraphs inside it, including those inside the `<div>`.

Not every website uses these tags in the same way. Inspect the page before deciding what to extract.

**Inspect and View page source:** Inspect shows the browser's current page structure, which JavaScript may have changed. Right-click the page and choose **View page source** to see the HTML originally received by the browser. Our Python code parses its own downloaded response and does not run JavaScript, so the structures can differ.

Reference: [Inspecting elements in Chrome DevTools](https://developer.chrome.com/docs/devtools/dom).

In [ ]:
soup = BeautifulSoup(response.content, "html.parser")
title = soup.find("title")
if title is not None:
    print(title.get_text(" ", strip=True))

### Choose a part of the page

We first look for `<main>`, then `<article>`, then `<body>`. `or` takes the first available element. If none exists, we use the whole parsed page.

This is a starting recipe. Some websites need a different tag or a more specific selection.

In [ ]:
content = soup.find("main") or soup.find("article") or soup.find("body") or soup
print(content.name)

Remove scripts, styles, navigation and headers or footers inside that selection. `.decompose()` removes an element. This reduces some noise; it does not guarantee that all remaining text is relevant.

In [ ]:
for element in content.find_all(["script", "style", "noscript", "nav", "header", "footer"]):
    element.decompose()

paragraphs = content.find_all("p")
print(len(paragraphs))

`find_all("p")` gives us paragraph elements. For each one, `get_text(" ", strip=True)` removes the tags, separates text pieces with spaces, and trims surrounding spaces.

We use the same pattern as Lecture 7: make an empty list, loop, append.

In [ ]:
# Create an empty list called texts.
# Loop through paragraphs, get each paragraph's text, and append it if it is not empty.


<details>
<summary>Show solution</summary>

```python
texts = []
for paragraph in paragraphs:
    paragraph_text = paragraph.get_text(" ", strip=True)
    if paragraph_text:
        texts.append(paragraph_text)

print(len(texts))
```



</details>

Join the paragraphs into a single string. `"\n"` puts a line break between them, as in the speeches from Lecture 7.

In [ ]:
page_text = "\n".join(texts)
print(len(page_text))
print(page_text[:1500])

### Check the content yourself

Read the text and compare it with the [SwissEnergy programme page](https://www.bfe.admin.ch/en/the-swissenergy-programme) in your browser.

Does it contain the programme description? Is it in the expected language? Are there contact details or unrelated paragraphs mixed in? Are important parts missing?

A long string can still be a cookie notice or an access challenge. A short string can be a complete short page. Length helps us find cases to inspect; it does not prove success.

## 5. Reuse our work on one page

Before we add a loop, we turn our existing extraction steps into a function. We used functions in Lecture 3: `def` gives a piece of code a name, and `return` sends its result back.

`extract_text(html)` takes the downloaded HTML and returns a string. The code inside is the same paragraph extraction we just ran, with indentation added. It does not download anything.

In [ ]:
def extract_text(html):
    soup = BeautifulSoup(html, "html.parser")
    content = soup.find("main") or soup.find("article") or soup.find("body") or soup

    for element in content.find_all(["script", "style", "noscript", "nav", "header", "footer"]):
        element.decompose()

    texts = []
    for paragraph in content.find_all("p"):
        paragraph_text = paragraph.get_text(" ", strip=True)
        if paragraph_text:
            texts.append(paragraph_text)

    return "\n".join(texts)

### Try the function on the page we already downloaded

There is no new request here. We reuse `response.content` from section 3.

**Predict:** should the beginning of the text match the result from section 4?

In [ ]:
test_text = extract_text(response.content)
print(test_text[:500])

### What if a download fails?

We introduce one new idea: `try` and `except`. Python attempts the code under `try`. If a request fails, it runs the code under `except` instead of stopping the notebook.

The function below still works on **only one URL**. It returns a dictionary containing the URL, HTTP status, extracted text, any error, and an extraction status.

Read it in three parts:

1. Start a record for this URL.
2. Try to download it and reuse `extract_text()`.
3. If the request fails, record the error. Return the record either way.

`raise_for_status()` makes HTTP responses such as 403 count as request failures too. The HTTP column lets us distinguish them from timeouts, where no response arrived.

In [ ]:
def scrape_page(url):
    result = {"url": url, "http_status": None, "text": "", "error": ""}

    try:
        response = requests.get(url, timeout=20)
        result["http_status"] = response.status_code
        response.raise_for_status()

        if "html" not in response.headers.get("Content-Type", "").lower():
            result["status"] = "not HTML"
        else:
            result["text"] = extract_text(response.content)
            if result["text"]:
                result["status"] = "text found: check"
            else:
                result["status"] = "no paragraphs"

    except requests.RequestException as error:
        result["status"] = "request failed"
        result["error"] = str(error)

    return result

### Test one URL before repeating

We use the same Swiss Federal Office of Energy page. Inspect the dictionary one field at a time before moving on.

In [ ]:
one_result = scrape_page(urls[2])
print(one_result["http_status"])
print(one_result["status"])
print(one_result["text"][:500])
print(one_result["error"])

### Now the loop is just repetition

We already know how to handle one page. The loop only calls that function, collects the result and shows progress.

**Predict:** if the fourth request fails, will the fifth URL still be attempted? Where is the failure handled?

In [ ]:
results = []
for url in urls:
    result = scrape_page(url)
    results.append(result)
    if result["status"] == "text found: check":
        print("✅", result["status"], url)
    else:
        print("❌", result["status"], url)
    time.sleep(1)

Every call returns a record, including a failed request, so every URL stays in `results`.

`time.sleep(1)` pauses between requests. Inspect the stored results below without rerunning this loop.

`"text found: check"` means the extractor found paragraph text. We still need to read it before deciding whether it is useful. There is no minimum length that proves success.

The progress messages use ✅ when paragraph text was extracted and ❌ when the request failed or no paragraph text was extracted. Still read the extracted text before marking it usable.

In [ ]:
scraping_results = pd.DataFrame(results)
scraping_results = sources.merge(scraping_results, left_on="Url", right_on="url", how="left")
scraping_results["character_count"] = scraping_results["text"].str.len()
scraping_results["review"] = "not reviewed"
scraping_results[["Title", "http_status", "status", "character_count", "review"]]

## 6. Diagnose the results

**Predict:** did all ten URLs produce useful text? Do all the HTTP 200 responses have paragraphs?

These are live websites. Results can change between computers and dates. The preparation test found usable paragraph text on several pages, access refusals on others, and an empty paragraph selection on BKW.

In [ ]:
# Count the values in the status column.


<details>
<summary>Show solution</summary>

```python
scraping_results["status"].value_counts()
```



</details>

### Look at the failures

Keep the URL and the error together. Missing text from an unavailable page is **missing data**, not evidence that the website never discusses energy.

In [ ]:
needs_attention = scraping_results[scraping_results["status"] != "text found: check"]
needs_attention[["Title", "http_status", "status", "error"]]

### Read one candidate before calling it usable

Choose a row that contains text. Read the extracted content and compare it with the browser page. Only after this check should you mark its `review` as `"usable"`, `"partial"`, or `"unusable"`.

Check the language, whether the intended content is present, and whether navigation, consent notices or other unwanted material dominate.

In [ ]:
candidates = scraping_results[scraping_results["character_count"] > 0]
if candidates.empty:
    print("No text to inspect. Read the error table above.")
else:
    row_index = candidates.index[0]
    print("Row:", row_index)
    print(scraping_results.loc[row_index, "Title"])
    print(scraping_results.loc[row_index, "text"][:2000])

After comparing a chosen row with its source page, record your judgement in a cell like this. Replace the index and judgement with the ones you checked:

```python
scraping_results.loc[2, "review"] = "usable"
```

Do not run this example until you have checked that row. A table of download statuses and a table of reviewed content answer different questions.

### Why can a page look fine in a browser but fail here?

| Observation | What to investigate |
|---|---|
| HTTP 403 | Access was refused. A browser may pass checks that this request cannot. |
| HTTP 404 | Check the address; the page may have moved or been removed. |
| Timeout or connection error | Check the network and whether the site is available. |
| HTTP 200, no paragraphs | The selected container or paragraph tag may not match the page structure. Content might also require JavaScript. |
| Text found, wrong content | Inspect the title, language and text; the extractor may have collected unrelated material. |

Beautiful Soup parses the HTML it receives. It does not open a browser, execute JavaScript, or automatically identify the content you intended.

## 7. Save all results

We keep the extracted text, the source URLs, errors and review decisions together. Failed rows remain visible when we return to the file.

In [ ]:
scraping_results.to_csv("scraping_results.csv", index=False)

## 8. Your turn

1. Inspect one row with paragraph text. Compare it with the browser and record your review.
2. Inspect one failed row. Explain whether the failure happened during the request or during content extraction.
3. Find a row with HTTP 200. Explain why that status alone does not establish success.

### Optional: investigate the BKW selection

Use the second URL to download BKW once. Inspect paragraph text in the whole page and in `<main>` separately. You can reuse the code from sections 3 and 4.

```python
bkw_response = requests.get(urls[1], timeout=20)
bkw_response.raise_for_status()
bkw_soup = BeautifulSoup(bkw_response.content, "html.parser")
print(len(bkw_soup.find_all("p")))
bkw_main = bkw_soup.find("main")
if bkw_main is not None:
    print(len(bkw_main.find_all("p")))
```

If the whole page has paragraphs but `<main>` does not, does searching the whole page solve the problem? Read those paragraphs before deciding: they might all be footer text. Some websites use custom HTML elements for the content we wanted.

## What we did

| Task | Code |
|---|---|
| Make a URL list | `sources["Url"].tolist()` |
| Download a page | `requests.get(url, timeout=20)` |
| Check for HTTP errors | `response.raise_for_status()` |
| Parse HTML | `BeautifulSoup(response.content, "html.parser")` |
| Find paragraph elements | `content.find_all("p")` |
| Extract plain text | `paragraph.get_text(" ", strip=True)` |
| Keep working after a request fails | `try` and `except requests.RequestException` |
| Reuse the extraction steps | `extract_text(html)` |
| Attempt one page and record failures | `scrape_page(url)` |
| Repeat for every URL | `result = scrape_page(url)` inside a short loop |
| Check actual content | Read it and compare it with the source page |

The output is a collection of attempted extractions with an audit trail. Reviewed text can later be cleaned and analysed using the methods from Lecture 7.